# ID Document Fraud Detection

Binary classifier that flags fraudulent identity documents (driver's licenses and national ID cards) for the **FreUID Challenge 2026 (IJCAI-ECAI)**.

**Key idea:** instead of a random split, the model is validated on a *document type it never saw during training*. In production, a fraud detector constantly meets new card designs, so this is the setting that matters.

| | |
|---|---|
| Data | 69,352 images, 5 document types, ~42% fraudulent |
| Model | EfficientNet-B0 (ImageNet weights), fine-tuned end to end |
| Validation | Leave-one-document-type-out (held out: `GUINEA/DL`) |
| Best epoch | ROC-AUC **0.999**, recall **0.999**, accuracy **0.82** at threshold 0.5 |

## 1. Data

Each row has an image path, a binary `label` (1 = fraudulent), an `is_digital` flag and the document `type` (country / document kind).

In [1]:
import os
import pandas as pd

base = '/kaggle/input/datasets/coreclock/the-freuid-challenge-2026-ijcai-ecai-mine'
full_labels = pd.read_csv(f'{base}/train_labels.csv')

print(full_labels.shape)
print(full_labels.head(10))
print(full_labels.columns.tolist())

(69352, 5)
                                 id  \
0  0000e24edb864abe8b0defa703742dcb   
1  000447addd8a4e4aabb62238ba3d559f   
2  0004b435792e4a5387a228133ad27a5f   
3  000514f0340642f3b2eb83bff862458a   
4  0006fe32740941c58a2a96cf28768e7f   
5  0007db19385841f8b45d50329be83a1a   
6  00083e9c305b467b9a414eb96c05e877   
7  0008b444e8514df2aa71a47a9df69ce7   
8  0008d2cb7c64466d975406593da2c719   
9  0009a1b4b8f04e2bb920e791d087165a   

                                    image_path  label  is_digital  \
0  train/0000e24edb864abe8b0defa703742dcb.jpeg      1        True   
1  train/000447addd8a4e4aabb62238ba3d559f.jpeg      0        True   
2  train/0004b435792e4a5387a228133ad27a5f.jpeg      1        True   
3  train/000514f0340642f3b2eb83bff862458a.jpeg      0        True   
4  train/0006fe32740941c58a2a96cf28768e7f.jpeg      0        True   
5  train/0007db19385841f8b45d50329be83a1a.jpeg      1        True   
6  train/00083e9c305b467b9a414eb96c05e877.jpeg      0        True   
7  trai

## 2. Validation strategy: hold out an entire document type

A random split would put images of the same card design in both train and validation, and the model could score well just by memorising each template. Holding out a whole document type measures whether it learned *fraud cues* that transfer to an unseen design.

The cell below scores every 1- and 2-type hold-out by how close it gets to a 20% validation size and to the overall fraud rate, then uses the best candidate.

In [3]:
import itertools

sample_df = full_labels.copy()

target_val_ratio = 0.20
overall_pos_rate = sample_df['label'].mean()
all_types = sorted(sample_df['type'].unique())

rows = []

for r in [1, 2]:
    for val_types in itertools.combinations(all_types, r):
        val_mask = sample_df['type'].isin(val_types)
        val_df_tmp = sample_df[val_mask]
        train_df_tmp = sample_df[~val_mask]

        if len(val_df_tmp) == 0 or len(train_df_tmp) == 0:
            continue

        val_ratio = len(val_df_tmp) / len(sample_df)
        val_pos_rate = val_df_tmp['label'].mean()
        score = abs(val_ratio - target_val_ratio) + abs(val_pos_rate - overall_pos_rate)

        rows.append({
            'val_types': val_types,
            'val_size': len(val_df_tmp),
            'val_ratio': round(val_ratio, 4),
            'val_pos_rate': round(val_pos_rate, 4),
            'score': round(score, 4)
        })

candidates = pd.DataFrame(rows).sort_values('score')
print(candidates.head(10))

# Best-scoring hold-out from the table above
chosen_val_types = ('GUINEA/DL',)

val_df = sample_df[sample_df['type'].isin(chosen_val_types)].reset_index(drop=True)
train_df = sample_df[~sample_df['type'].isin(chosen_val_types)].reset_index(drop=True)

print(train_df.shape, val_df.shape)
print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print("Train positive rate:", train_df['label'].mean())
print("Val positive rate:", val_df['label'].mean())
print("Train types:", train_df['type'].nunique())
print("Val types:", val_df['type'].nunique())
print("Overlap types:", len(set(train_df['type']).intersection(set(val_df['type']))))

                     val_types  val_size  val_ratio  val_pos_rate   score
2                 (GUINEA/DL,)     13389     0.1931        0.4024  0.0277
0                  (BENIN/DL,)     13369     0.1928        0.4015  0.0289
4             (MOZAMBIQUE/DL,)     13365     0.1927        0.4013  0.0291
3              (MAURITIUS/ID,)     13362     0.1927        0.4011  0.0294
1                  (EGYPT/DL,)     15867     0.2288        0.4958  0.1014
6        (BENIN/DL, GUINEA/DL)     26758     0.3858        0.4020  0.2070
13  (GUINEA/DL, MOZAMBIQUE/DL)     26754     0.3858        0.4019  0.2070
12   (GUINEA/DL, MAURITIUS/ID)     26751     0.3857        0.4018  0.2071
8    (BENIN/DL, MOZAMBIQUE/DL)     26734     0.3855        0.4014  0.2072
7     (BENIN/DL, MAURITIUS/ID)     26731     0.3854        0.4013  0.2073
(55963, 5) (13389, 5)
label
0    32004
1    23959
Name: count, dtype: int64
label
0    8001
1    5388
Name: count, dtype: int64
Train positive rate: 0.4281221521362329
Val positive rate:

## 3. Dataset and augmentation

Images are resized to 224×224 and normalised with ImageNet statistics. Training adds light colour jitter to mimic differences in scanning and lighting.

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class DocDataset(Dataset):
    def __init__(self, df, root, transform=None):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['image_path'].split('/')[-1]
        img_path = os.path.join(self.root, 'train', 'train', filename)
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return img, label

In [8]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = DocDataset(train_df, base, train_tfms)
val_ds = DocDataset(val_df, base, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

x, y = next(iter(train_loader))
print(x.shape, y.shape)

torch.Size([64, 3, 224, 224]) torch.Size([64])


## 4. Model

EfficientNet-B0 pretrained on ImageNet, with the classifier head replaced by a single logit. Trained with `BCEWithLogitsLoss`, Adam (lr 1e-4) and mixed precision on a Tesla T4.

In [9]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 1)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 100MB/s]


## 5. Training and evaluation

Besides accuracy, validation tracks recall (missed fraud is the costly error), F1, ROC-AUC and PR-AUC. The AUC metrics are threshold-free, so they show how well the model *ranks* fraud above genuine documents regardless of where the decision cutoff sits.

In [10]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, average_precision_score
)

scaler = torch.amp.GradScaler('cuda')

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)

            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_probs.extend(probs.cpu().numpy().flatten().tolist())
            all_preds.extend(preds.cpu().numpy().flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())

    val_loss = running_loss / len(loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)
    val_prec = precision_score(all_labels, all_preds, zero_division=0)
    val_rec = recall_score(all_labels, all_preds, zero_division=0)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)

    if len(set(all_labels)) > 1:
        val_roc_auc = roc_auc_score(all_labels, all_probs)
        val_pr_auc = average_precision_score(all_labels, all_probs)
    else:
        val_roc_auc = float("nan")
        val_pr_auc = float("nan")

    cm = confusion_matrix(all_labels, all_preds)
    return val_loss, val_acc, val_prec, val_rec, val_f1, val_roc_auc, val_pr_auc, cm

In [12]:
for epoch in range(5):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_prec, val_rec, val_f1, val_roc_auc, val_pr_auc, cm = validate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/5")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"Val Recall: {val_rec:.4f}")
    print(f"Val F1: {val_f1:.4f}")
    print(f"Val ROC-AUC: {val_roc_auc:.4f}")
    print(f"Val PR-AUC: {val_pr_auc:.4f}")

Epoch 1/5
Train Loss: 0.1145
Val Loss: 1.1067
Val Acc: 0.5444
Val Recall: 0.9998
Val F1: 0.6385
Val ROC-AUC: 0.9958
Val PR-AUC: 0.9945
Epoch 2/5
Train Loss: 0.0111
Val Loss: 0.4002
Val Acc: 0.8144
Val Recall: 0.9989
Val F1: 0.8124
Val ROC-AUC: 0.9986
Val PR-AUC: 0.9984
Epoch 3/5
Train Loss: 0.0055
Val Loss: 0.4334
Val Acc: 0.8178
Val Recall: 0.9985
Val F1: 0.8152
Val ROC-AUC: 0.9979
Val PR-AUC: 0.9974
Epoch 4/5
Train Loss: 0.0051
Val Loss: 0.3899
Val Acc: 0.8207
Val Recall: 0.9987
Val F1: 0.8176
Val ROC-AUC: 0.9986
Val PR-AUC: 0.9984
Epoch 5/5
Train Loss: 0.0030
Val Loss: 0.7081
Val Acc: 0.7181
Val Recall: 0.9991
Val F1: 0.7404
Val ROC-AUC: 0.9978
Val PR-AUC: 0.9973


## 6. Results

Validation on the held-out `GUINEA/DL` set (13,389 images). Precision is derived from the logged F1 and recall.

| Epoch | Val loss | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|---|
| 1 | 1.107 | 0.544 | 0.469 | 1.000 | 0.639 | 0.996 | 0.995 |
| 2 | 0.400 | 0.814 | 0.685 | 0.999 | 0.812 | 0.999 | 0.998 |
| 3 | 0.433 | 0.818 | 0.689 | 0.999 | 0.815 | 0.998 | 0.997 |
| **4** | **0.390** | **0.821** | **0.692** | **0.999** | **0.818** | **0.999** | **0.998** |
| 5 | 0.708 | 0.718 | 0.588 | 0.999 | 0.740 | 0.998 | 0.997 |

![Validation metrics per epoch](assets/validation_metrics.png)

### What the numbers say

- **Ranking is near perfect on an unseen document type.** ROC-AUC ≈ 0.999 means almost every fraudulent card scores higher than almost every genuine one.
- **The 0.5 threshold is the bottleneck.** Recall is ~1.0 but precision is ~0.69: at epoch 4 roughly 30% of genuine Guinean licenses are flagged as fraud. The model's scores shift upward on an unfamiliar design, so the default cutoff sits in the wrong place. The ranking is right; the calibration is not.
- **Overfitting after epoch 4.** Training loss falls to 0.003 while validation loss jumps from 0.39 to 0.71 at epoch 5.

## 7. Limitations and next steps

- **Threshold / calibration:** choose the cutoff on the training types (e.g. out-of-fold predictions) or calibrate scores per document type, and report accuracy at that threshold.
- **Shortcut check (done):** `shortcut_check.py` trains a model on file metadata only (image size, file size, JPEG quantisation tables). On each held-out document type it scores close to chance (ROC-AUC 0.36–0.63; GUINEA/DL 0.46), so the CNN's 0.999 is not explained by file artefacts.
- **Full cross-validation:** repeat with each of the 5 document types held out and report mean ± std (`EGYPT/DL` has a higher fraud rate, 0.50 vs ~0.40).
- **Training hygiene:** keep the best checkpoint, add early stopping and a learning-rate schedule.
- **Resolution:** 224×224 squashes the card's aspect ratio and fine detail where tampering shows up; try higher resolution or tiles. Drop horizontal flips, since mirrored text never occurs on real documents.